### **Mount Drive and Create Folders**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os

os.makedirs("/content/drive/MyDrive/project/tableau", exist_ok=True)
os.makedirs("/content/drive/MyDrive/project/data", exist_ok=True)

print("Folders created successfully")

Folders created successfully


### **Create Spark Session**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("URL_Malicious_Detection") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


### **Load All LIBSVM Files**

In [3]:
data_path = "/content/drive/MyDrive/data/url_svmlight/*"

In [4]:
data = spark.read.format("libsvm").load(data_path)

### **Basic Data Inspection**

In [5]:
print("Total records:", data.count())
data.printSchema()

Total records: 2396194
root
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)



In [6]:
data.show(5)

+-----+--------------------+
|label|            features|
+-----+--------------------+
| -1.0|(3231961,[3,4,5,1...|
| -1.0|(3231961,[1,3,4,5...|
| -1.0|(3231961,[1,3,4,5...|
| -1.0|(3231961,[3,4,5,1...|
|  1.0|(3231961,[1,3,4,5...|
+-----+--------------------+
only showing top 5 rows


### **Check Class Distribution**

In [10]:
from pyspark.sql.functions import col
data = data.filter((col("label") == -1) | (col("label") == 1))

class_dist = data.groupBy("label").count()

class_dist.show()

class_dist.toPandas().to_csv(
    "/content/drive/MyDrive/project/tableau/class_distribution.csv",
    index=False
)

+-----+-------+
|label|  count|
+-----+-------+
| -1.0|1603985|
|  1.0| 792145|
+-----+-------+



### **Check Missing Values**

In [11]:
from pyspark.sql.functions import isnan, when, count, col

missing = data.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in data.columns
])

missing.show()

+-----+--------+
|label|features|
+-----+--------+
|    0|       0|
+-----+--------+



### **Partition Optimization**

In [12]:
data = data.repartition(200)

print("Partitions:", data.rdd.getNumPartitions())

Partitions: 200


### **Save Clean Data as Parquet**

In [13]:
data.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/project/data/url_clean.parquet"
)

### **Sample Data for Tableau Dashboard**

In [14]:
sample = data.sample(fraction=0.01, seed=42)

sample.toPandas().to_csv(
    "/content/drive/MyDrive/project/tableau/data_sample.csv",
    index=False
)